# Coastal NC High-Ground Finder — Stage 1: Elevation**Status: stage 1 runnable** (July 2026). Sibling to `pueblo_bonito_lidar_poc.ipynb`;Sections 1-3 ported from it.## Scope of this stageFind the high ground. Nothing else. No parcels, no listings, no prices.Carteret County averages 7 ft above sea level; the community named Sea Level floods~75% of the time a hurricane passes. County averages say give up. Relict beach ridgesand marine terraces don't appear in a county average — finding them is the point.## Processing unit: counties. Filter: distance-to-coast, scored not gated.The whole Duck-to-Wilmington AOI at 10m is a 23,210 x 27,642 grid — 642M cells,2.6 GB as float32, and `find_high_ground` holds 3-4 derived arrays at once, so a~10 GB peak working set. That won't survive a laptop.Tiled by county the largest tile is Carteret at 79M cells (0.32 GB, ~1.3 GB peak).Everything downstream — parcels, NFHL, permits, tax records — is county-keyed anyway,so the tiling and the join key are the same thing.Distance-to-coast is computed as a **raster layer and kept as a column**, never usedto clip. Same rule as access scoring: score, don't gate. A hard 30-mile buffer wouldbehave completely differently in Dare (still in the sounds at 30 mi) than in Brunswick(well inland), and it would silently delete the boat-access scenario.## Data note3DEP 1/3 arc-second (~10m) is a lidar-*derived* product, not a point cloud. In coastalNC it is largely built from the NC Floodplain Mapping Program QL2 lidar, which is oneof the better statewide collections in the country. 10m is the right screeningresolution here; the QL2 source is available at ~1m for finalists in stage 2.## Setup```python3 -m venv .venv && source .venv/bin/activatepip install -r requirements.txtcp .env.example .env      # then fill in keys```

In [ ]:
# --- README export: cell 0 of this notebook IS the project README -----------import jsonfrom pathlib import PathNB_FILE = Path("nc_highground.ipynb")cells = json.loads(NB_FILE.read_text())["cells"]header_md = "".join(cells[0]["source"])banner = ("<!-- AUTO-GENERATED from the header cell of "          f"{NB_FILE.name} - edit there, not here. -->\n\n")Path("README.md").write_text(banner + header_md + "\n")print(f"[ok] README.md written ({len(header_md)} chars from notebook cell 0)")print("[note] reads the SAVED notebook - hit save before running this cell")

## 1 · Imports

In [ ]:
import os, json, warningsfrom pathlib import Pathimport numpy as npimport pandas as pdimport rasteriofrom rasterio.transform import Affineimport matplotlib.pyplot as pltfrom matplotlib.colors import ListedColormap, BoundaryNormfrom matplotlib.patches import Patchfrom scipy import ndimagefrom dotenv import load_dotenv%matplotlib inlineplt.rcParams["figure.dpi"] = 110warnings.filterwarnings("ignore", category=UserWarning)

## 2 · Environment & configuration

In [ ]:
# --- .env ---------------------------------------------------------------# None of stage 1 requires a key: py3dep / The National Map are open.# These are read now so a missing key fails here rather than 40 minutes in.load_dotenv()KEYS = {    "OPENTOPOGRAPHY_API_KEY": "only if using OpenTopography instead of py3dep",    "ORS_API_KEY":            "OpenRouteService - drive times, stage 3",    "REGRID_API_KEY":         "parcels, stage 4 (NC OneMap is free, this is a fallback)",}for k, why in KEYS.items():    v = os.getenv(k)    print(f"  {'[ok] ' if v else '[--] '}{k:<26} {'set' if v else 'not set'}   ({why})")print("\nStage 1 needs none of these.")

In [ ]:
# --- Config -------------------------------------------------------------DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)DEM_DIR  = DATA_DIR / "dem"; DEM_DIR.mkdir(exist_ok=True)OUT_DIR  = Path("output"); OUT_DIR.mkdir(exist_ok=True)# Coastal counties, Duck -> Wilmington, mainland side.# Ordered north to south. Comment lines out to shrink a run.COUNTIES = [    "Currituck", "Camden", "Pasquotank", "Perquimans",   # far north, mostly sound    "Dare", "Tyrrell", "Washington", "Hyde",             # behind Roanoke Island    "Beaufort", "Pamlico", "Craven",                     # Neuse / Pamlico    "Carteret", "Jones", "Onslow",                       # Beaufort / Cape Lookout reach    "Pender", "New Hanover", "Brunswick",                # Wilmington]STATE_FIPS = "37"          # North CarolinaTARGET_RES_M = 10.0        # 3DEP 1/3 arc-secondWORKING_EPSG = 32618       # UTM 18N, meters - all area math happens here# Elevation bands (FEET, NAVD88). Hard breaks; never a continuous ramp.BAND_EDGES_FT = [0, 5, 10, 15, 20, 25, 100]BAND_COLORS = ["#8B0000",  # 0-5    underwater in a bad storm               "#E8722C",  # 5-10   the in-laws' situation               "#F2D338",  # 10-15  marginal               "#A8D96B",  # 15-20  baseline               "#2E8B57",  # 20-25  target               "#0B4F6C"]  # 25+    idealBAND_LABELS = ["0-5 ft", "5-10 ft", "10-15 ft", "15-20 ft", "20-25 ft", "25+ ft"]THRESHOLD_SWEEP_FT = [10, 15, 18, 20, 22, 25, 30]PRIMARY_THRESHOLD_FT = 20.0MIN_REGION_ACRES = 10.0     # the 10-20 acre targetMIN_PAD_ACRES    = 2.0      # house + pond + septic + setbacksM_TO_FT = 3.28084SQM_PER_ACRE = 4046.86print(f"{len(COUNTIES)} counties queued, {TARGET_RES_M:.0f}m cells, "      f"threshold sweep {THRESHOLD_SWEEP_FT}")

## 3 · Core functionsBanding, region finding, buildable-pad test, distance-to-water.

In [ ]:
# --- Core functions -----------------------------------------------------def to_feet(dem_meters):    # Meters (NAVD88) -> feet. The ONLY place this conversion happens.    return dem_meters * M_TO_FTdef banded_cmap():    cmap = ListedColormap(BAND_COLORS)    cmap.set_bad("#DDDDDD")      # nodata / water    cmap.set_under("#4A4A4A")    # below 0 ft NAVD88    return cmap, BoundaryNorm(BAND_EDGES_FT, cmap.N, clip=False)def show_banded(arr_ft, title, outlines=None, ax=None, save=None, figsize=(13, 10)):    cmap, norm = banded_cmap()    if ax is None:        _, ax = plt.subplots(figsize=figsize)    ax.imshow(arr_ft, cmap=cmap, norm=norm, interpolation="nearest")    if outlines is not None and np.any(outlines):        ax.contour(outlines > 0, levels=[0.5], colors="white", linewidths=0.9)    ax.set_title(title); ax.axis("off")    ax.legend(handles=[Patch(facecolor=c, label=l)                       for c, l in zip(BAND_COLORS, BAND_LABELS)],              loc="lower left", framealpha=0.9, fontsize=8,              title="Elevation (ft NAVD88)")    if save:        plt.savefig(OUT_DIR / save, dpi=180, bbox_inches="tight")    return axdef find_high_ground(dem_ft, threshold_ft, cell_m,                     min_acres=MIN_REGION_ACRES, connectivity=2):    # Contiguous patches >= threshold_ft and >= min_acres.    # Returns (labels, regions) with regions sorted by acreage desc.    cell_area = cell_m ** 2    min_cells = int(np.ceil(min_acres * SQM_PER_ACRE / cell_area))    mask = np.nan_to_num(dem_ft, nan=-9999) >= threshold_ft    structure = ndimage.generate_binary_structure(2, connectivity)    labels, n = ndimage.label(mask, structure=structure)    if n == 0:        return labels, []    counts = np.bincount(labels.ravel()); counts[0] = 0    keep = np.where(counts >= min_cells)[0]    if keep.size == 0:        return np.zeros_like(labels), []    objs = ndimage.find_objects(labels)    regions = []    for lab in keep:        sl = objs[lab - 1]        sub = labels[sl] == lab        vals = dem_ft[sl][sub]        rows, cols = np.nonzero(sub)        regions.append({            "label":   int(lab),            "acres":   float(counts[lab] * cell_area / SQM_PER_ACRE),            "mean_ft": float(np.nanmean(vals)),            "max_ft":  float(np.nanmax(vals)),            "row_c":   float(rows.mean() + sl[0].start),            "col_c":   float(cols.mean() + sl[1].start),            "slice":   sl,        })    regions.sort(key=lambda r: r["acres"], reverse=True)    return np.where(np.isin(labels, keep), labels, 0), regionsdef buildable_pad(region_mask, cell_m, pad_acres=MIN_PAD_ACRES):    # Largest inscribed circle. Catches ribbon-shaped terraces that pass on    # acreage alone but have nowhere to put a house.    dt = ndimage.distance_transform_edt(region_mask, sampling=cell_m)    r_m = float(dt.max())    acres = np.pi * r_m ** 2 / SQM_PER_ACRE    return {"pad_radius_m": r_m, "pad_acres": acres, "pad_fits": acres >= pad_acres}def distance_to_water_km(dem_ft, cell_m, water_level_ft=1.0):    # Distance from every cell to the nearest tidal-water cell, in km.    # Proxy: cells at or below water_level_ft. Crude but it needs no shoreline    # download and it captures sounds, rivers and ocean alike - which is right,    # because surge comes up the sounds, not just off the beach.    water = np.nan_to_num(dem_ft, nan=0.0) <= water_level_ft    if not water.any():        return np.full(dem_ft.shape, np.nan, dtype="float32")    dt = ndimage.distance_transform_edt(~water, sampling=cell_m)    return (dt / 1000.0).astype("float32")

## 4 · County boundaries

In [ ]:
# --- County boundaries --------------------------------------------------# TIGER/Line via pygris. Cached to disk after the first pull.import geopandas as gpdCO_PATH = DATA_DIR / "nc_counties.gpkg"if CO_PATH.exists():    co = gpd.read_file(CO_PATH)else:    import pygris    co = pygris.counties(state="NC", cb=True, cache=True)    co = co.rename(columns={"NAME": "county"})[["county", "geometry"]]    co.to_file(CO_PATH, driver="GPKG")co = co[co["county"].isin(COUNTIES)].reset_index(drop=True)missing = set(COUNTIES) - set(co["county"])if missing:    print(f"[warn] not matched: {sorted(missing)}")co_utm = co.to_crs(WORKING_EPSG)co["area_km2"] = co_utm.area / 1e6print(co[["county", "area_km2"]].sort_values("area_km2", ascending=False).to_string(index=False))print(f"\ntotal {co['area_km2'].sum():,.0f} km2 across {len(co)} counties")

## 5 · Fetch the 10m DEMsOne GeoTIFF per county, cached. Re-runs are free.

In [ ]:
# --- DEM fetch, one county at a time, cached ----------------------------# py3dep pulls 3DEP 1/3 arc-second and returns an xarray DataArray.# Cached as GeoTIFF so a re-run costs nothing.import py3depimport rioxarray  # noqa: F401  (registers the .rio accessor)def fetch_county_dem(name, geom, res=int(TARGET_RES_M), overwrite=False):    out = DEM_DIR / f"{name.replace(' ', '_')}_{res}m.tif"    if out.exists() and not overwrite:        return out    print(f"  fetching {name} ...", end="", flush=True)    da = py3dep.get_dem(geom, resolution=res, crs=4326)    da = da.rio.reproject(WORKING_EPSG, resolution=res)    da.rio.to_raster(out, compress="lzw")    print(f" {out.stat().st_size/1e6:.0f} MB")    return outdem_paths = {}for _, row in co.iterrows():    dem_paths[row["county"]] = fetch_county_dem(row["county"], row["geometry"])print(f"\n[ok] {len(dem_paths)} county DEMs on disk, "      f"{sum(p.stat().st_size for p in dem_paths.values())/1e9:.2f} GB total")

## 6 · ProcessOne county in memory at a time — this is why the tiling exists.

In [ ]:
# --- Per-county processing loop -----------------------------------------# One county in memory at a time. Largest tile (Carteret) is ~79M cells.def load_dem_ft(path):    with rasterio.open(path) as src:        dem = src.read(1).astype("float32")        prof, tf, crs = src.profile, src.transform, src.crs        cell = float(abs(src.transform.a))    nd = prof.get("nodata")    if nd is not None:        dem[dem == nd] = np.nan    dem[dem < -100] = np.nan          # 3DEP sentinels    # The assertion that actually carried over from Chaco: there, the endpoint    # silently served 10m when we ordered 1m. Fail loudly rather than quietly.    assert abs(cell - TARGET_RES_M) < TARGET_RES_M * 0.25, (        f"{path.name}: expected ~{TARGET_RES_M}m cells, got {cell:.2f}m")    return to_feet(dem), cell, tf, crsrows, region_records = [], []for county, path in dem_paths.items():    dem_ft, cell_m, tf, crs = load_dem_ft(path)    valid = ~np.isnan(dem_ft)    dist_km = distance_to_water_km(dem_ft, cell_m)    rec = {"county": county,           "cells_M": valid.sum() / 1e6,           "mean_ft": float(np.nanmean(dem_ft)),           "p95_ft":  float(np.nanpercentile(dem_ft[valid], 95)) if valid.any() else np.nan,           "max_ft":  float(np.nanmax(dem_ft)) if valid.any() else np.nan}    for t in THRESHOLD_SWEEP_FT:        lb, regs = find_high_ground(dem_ft, t, cell_m)        n_build = 0        for r in regs:            sub = lb[r["slice"]] == r["label"]            pad = buildable_pad(sub, cell_m)            r.update(pad)            if pad["pad_fits"]:                n_build += 1            if t == PRIMARY_THRESHOLD_FT:                x, y = tf * (r["col_c"], r["row_c"])                region_records.append({                    "county": county, "threshold_ft": t,                    "acres": r["acres"], "mean_ft": r["mean_ft"],                    "max_ft": r["max_ft"], "pad_acres": r["pad_acres"],                    "pad_fits": r["pad_fits"], "x": x, "y": y,                    "dist_water_km": float(dist_km[int(r["row_c"]), int(r["col_c"])]),                })        rec[f"n{t}"] = len(regs)        rec[f"b{t}"] = n_build    rows.append(rec)    print(f"{county:<13} mean {rec['mean_ft']:5.1f} ft  max {rec['max_ft']:6.1f} ft  "          f"| >=20ft: {rec['n20']:>3} regions, {rec['b20']:>3} buildable")    del dem_ft, dist_kmsummary = pd.DataFrame(rows)regions_df = pd.DataFrame(region_records)summary.to_csv(OUT_DIR / "county_summary.csv", index=False)regions_df.to_csv(OUT_DIR / "regions_20ft.csv", index=False)print(f"\n[ok] {len(regions_df)} regions at {PRIMARY_THRESHOLD_FT:.0f} ft written")

## 7 · Results

In [ ]:
# --- Where is the high ground? ------------------------------------------cols = ["county", "mean_ft", "p95_ft", "max_ft"] + \       [c for t in THRESHOLD_SWEEP_FT for c in (f"n{t}", f"b{t}")]print(summary[cols].sort_values("b20", ascending=False).to_string(index=False))

In [ ]:
# --- Falloff curve, statewide -------------------------------------------tot_n = [summary[f"n{t}"].sum() for t in THRESHOLD_SWEEP_FT]tot_b = [summary[f"b{t}"].sum() for t in THRESHOLD_SWEEP_FT]fig, ax = plt.subplots(figsize=(9, 5))ax.plot(THRESHOLD_SWEEP_FT, tot_n, "o-", label=f"regions >= {MIN_REGION_ACRES:.0f} ac")ax.plot(THRESHOLD_SWEEP_FT, tot_b, "s-", label=f"of those, pad >= {MIN_PAD_ACRES:.0f} ac")ax.set_xlabel("Elevation threshold (ft NAVD88)")ax.set_ylabel("Count across all counties")ax.set_title("How fast the high ground runs out")ax.set_yscale("symlog"); ax.grid(alpha=0.3); ax.legend()plt.savefig(OUT_DIR / "threshold_sweep.png", dpi=200, bbox_inches="tight")plt.show()for t, n, b in zip(THRESHOLD_SWEEP_FT, tot_n, tot_b):    print(f"  {t:>3} ft -> {n:>5} regions, {b:>5} buildable")

In [ ]:
# --- Top regions at the primary threshold -------------------------------top = (regions_df[regions_df.pad_fits]       .sort_values(["pad_acres", "acres"], ascending=False)       .head(30))print(top[["county", "acres", "mean_ft", "max_ft",           "pad_acres", "dist_water_km"]].to_string(index=False))# dist_water_km is a COLUMN, not a filter. Read it against the boat model:# a region 3 km from tidal water is a short run to a ramp; 25 km inland is# safer from surge but the Cape Lookout trip stops being a morning activity.

## 8 · Maps

In [ ]:
# --- Banded map for the counties that actually have high ground ---------best = summary.sort_values("b20", ascending=False).head(4)["county"].tolist()print("rendering:", best)for county in best:    dem_ft, cell_m, tf, crs = load_dem_ft(dem_paths[county])    lb, regs = find_high_ground(dem_ft, PRIMARY_THRESHOLD_FT, cell_m)    show_banded(dem_ft,                f"{county} County - banded elevation, "                f">={PRIMARY_THRESHOLD_FT:.0f} ft outlined ({len(regs)} regions)",                outlines=lb,                save=f"banded_{county.replace(' ', '_')}.png")    plt.show()    del dem_ft, lb

## Stage 1 output- `output/county_summary.csv` — every county, mean/p95/max elevation, region counts  at each threshold- `output/regions_20ft.csv` — every high-ground region: acreage, buildable pad,  centroid, distance to tidal water- `output/banded_*.png` — the maps## Read the results before deciding anythingThe question stage 1 answers is not "which parcel" but **"does the high ground existat all, and where."** Three possible outcomes, and they point different directions:1. **Plenty of 20-ft ground near Beaufort** — tighten the threshold and move to parcels.2. **High ground exists but sits 40 km inland** — the drive-to-marina number becomes   the binding constraint and stage 3 access scoring decides everything.3. **Nothing clears 20 ft anywhere useful** — then the honest answer is that the   pond-fill strategy has to do more work than planned, or the search moves north   toward the Suffolk Scarp, or 18 ft becomes the number.## Stage 2 — validate before trustingRun the banded map against areas that actually flooded in Florence, Matthew andIsabel. They should land in the red and orange bands. This is the Chaco Figure 11check, and it happens before any realtor conversation, not after.## Stage 3 — access scoring, parallel not gatedDrive time to a road-accessible break; boat time from Beaufort / Harkers Island /Morehead City to Cape Lookout and the Core Banks; straight-line exposure to openocean. Kept as three columns. #1 and #2 want you close to water, #3 wants you far —the regions that score acceptably on all three are the answer.## Stage 4 — parcels, then listingsNC OneMap parcels clipped to surviving regions; NWI wetlands (pond permits);SSURGO soils (pond feasibility, septic); FEMA NFHL. Listings last, because theyare the only layer that changes weekly.